# Document Q&A (mini RAG) — Kaggle, no API key

Ask questions about one or more PDF files using entirely local, open-source models. The notebook creates embeddings with `all-MiniLM-L6-v2`, searches them with FAISS, and produces an answer with `Qwen/Qwen2.5-1.5B-Instruct`.

**No OpenAI, Hugging Face, or other API key is required.** Enable Internet only for the first run so Kaggle can download packages and public model weights. After saving the models as a Kaggle Dataset, you can run with Internet disabled.

## Kaggle setup

1. Create a new Kaggle Notebook and turn on a **GPU** accelerator (T4 is enough).
2. Create a Kaggle Dataset containing your PDF files, then attach it through **Add Input**.
3. Turn on Internet for this first run.
4. Run every cell in order. In the next cell, replace `YOUR_DATASET_FOLDER` with the folder name shown under `/kaggle/input/`.

Do not upload confidential documents to a public Kaggle dataset.

In [1]:
# Install the small libraries not guaranteed to be preinstalled in every Kaggle image.
!pip -q install -U sentence-transformers faiss-cpu pypdf accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.6 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import re
import numpy as np
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# Change this to your attached Kaggle Dataset folder.
DOCUMENT_FOLDER = Path('/kaggle/input/datasets/vaidiknakarani/project-list-3rd-year')
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
TOP_K = 4

assert torch.cuda.is_available(), 'Enable GPU in Kaggle: Settings → Accelerator → GPU.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## Read and chunk the PDFs

A chunk is a small overlapping piece of text. The overlap prevents important context from being split between chunks.

In [3]:
def extract_pdf_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            pages.append({'source': pdf_path.name, 'page': page_number, 'text': text})
    return pages

def chunk_text(text, chunk_size=900, overlap=160):
    words = text.split()
    step = chunk_size - overlap
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), step) if words[i:i + chunk_size]]

pdf_files = sorted(DOCUMENT_FOLDER.rglob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDFs found in {DOCUMENT_FOLDER}. Check DOCUMENT_FOLDER and attach your dataset.')

chunks = []
for pdf_file in pdf_files:
    for page_data in extract_pdf_pages(pdf_file):
        for text_chunk in chunk_text(page_data['text']):
            chunks.append({**page_data, 'text': text_chunk})

print(f'Loaded {len(pdf_files)} PDF(s) and created {len(chunks)} chunks.')
print('Example:', chunks[0]['text'][:300])

Loaded 1 PDF(s) and created 42 chunks.
Example: AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch AI/ML Capstone Project Problem Statements Parul University in collaboration with TelcoLearn B.Tech 3rd/4th Year — 2027 Graduating Batch — Phase 2 Bootcamp Arpit Tripathi & Sanjay Kumar Instructions for Students T ea


## Create the searchable vector index

The embedding model converts each chunk into numbers. FAISS finds chunks that are closest in meaning to a question.

In [4]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device='cuda')
embeddings = embedder.encode(
    [item['text'] for item in chunks],
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
).astype('float32')

index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product = cosine similarity after normalization
index.add(embeddings)
print(f'FAISS index contains {index.ntotal} chunks.')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index contains 42 chunks.


## Load the local language model

This downloads a public 1.5B-parameter model on the first run. It runs locally in the Kaggle session; your question is not sent to an API.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
).eval()
print('Model loaded.')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [6]:
def retrieve(question, top_k=TOP_K):
    query = embedder.encode([question], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(query, top_k)
    return [(chunks[i], float(score)) for i, score in zip(ids[0], scores[0])]

def answer_question(question, top_k=TOP_K, max_new_tokens=350):
    results = retrieve(question, top_k)
    context = '\n\n'.join(
        f'[{n}] Source: {item["source"]}, page {item["page"]}\n{item["text"]}'
        for n, (item, _) in enumerate(results, start=1)
    )
    messages = [
        {'role': 'system', 'content': 'Answer only from the supplied context. If the answer is not in the context, say: I could not find that in the documents. Cite sources like [1] or [2].'},
        {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer_tokens = generated[0][model_inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    print('Answer:\n', answer)
    print('\nRetrieved passages:')
    for n, (item, score) in enumerate(results, start=1):
        print(f'[{n}] {item["source"]}, page {item["page"]} (similarity {score:.3f})')
    return answer, results

# Replace with a question about your documents.
answer_question('What are the most important points in these documents?')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer:
 The most important points in these documents are:

1. **Legal Document Classification**: The projects aim to classify legal documents into different categories such as criminal, civil, constitutional, taxation, etc., using techniques like TF-IDF + Logistic Regression, XGBoost, LDA, Longformer Encoder-Decoder, and Sentence-BERT.

2. **Topic Discovery and Evolution**: Projects focus on discovering key legal themes across judgments over time using LDA and visualizing this through a decade timeline.

3. **Summarization and Question Answering**: There are efforts to fine-tune models for legal summarization and build question answering systems using techniques like Longformer Encoder-Decoder, Sentence-BERT, and Retrieval-Augmented Generation.

4. **Fake News Detection**: Some projects include mechanisms to detect fake news and provide fact-checks based on various datasets like kaggle: clmentbisaillon/fake-and-real-news-dataset and kaggle: mrisdal/fake-news.

5. **Performance Metrics

('The most important points in these documents are:\n\n1. **Legal Document Classification**: The projects aim to classify legal documents into different categories such as criminal, civil, constitutional, taxation, etc., using techniques like TF-IDF + Logistic Regression, XGBoost, LDA, Longformer Encoder-Decoder, and Sentence-BERT.\n\n2. **Topic Discovery and Evolution**: Projects focus on discovering key legal themes across judgments over time using LDA and visualizing this through a decade timeline.\n\n3. **Summarization and Question Answering**: There are efforts to fine-tune models for legal summarization and build question answering systems using techniques like Longformer Encoder-Decoder, Sentence-BERT, and Retrieval-Augmented Generation.\n\n4. **Fake News Detection**: Some projects include mechanisms to detect fake news and provide fact-checks based on various datasets like kaggle: clmentbisaillon/fake-and-real-news-dataset and kaggle: mrisdal/fake-news.\n\n5. **Performance Metr

## Optional: ask more questions

Run the next cell repeatedly with a new question. The model and index stay in memory, so later answers are faster.

### Troubleshooting
- **Out of memory:** change `LLM_MODEL` to `Qwen/Qwen2.5-0.5B-Instruct` or reduce `TOP_K` to 3.
- **No PDFs found:** correct `DOCUMENT_FOLDER`; the exact folder name appears under `/kaggle/input`.
- **Scanned PDF gives blank text:** OCR it before using this notebook, because `pypdf` reads embedded text rather than images.
- **Internet-off run:** save the model folders as a private Kaggle Dataset and set model paths to `/kaggle/input/<your-model-dataset>/<folder>`.

In [16]:
question = 'What dataset is used for the Chest X-Ray Pneumonia Detection project?'
answer_question(question)

Answer:
 The dataset used for the Chest X-Ray Pneumonia Detection project is available on Kaggle under the name "paultimothymooney/chest-xray-pneumonia".

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 3 (similarity 0.434)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 40 (similarity 0.388)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 2 (similarity 0.381)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 14 (similarity 0.337)


('The dataset used for the Chest X-Ray Pneumonia Detection project is available on Kaggle under the name "paultimothymooney/chest-xray-pneumonia".',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 3,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch Student Sub-problem T echnique App component A Extract HOG and pixel histogram features; train SVM and Random Forest baseline. Establish non-deep-learning accuracy ceiling. Supervised, feature engineering Backend baseline comparison tab B Train CNN Autoencoder on normal X-rays only; flag high reconstruction error as anomalous (no disease labels used). Unsupervised anomaly detection Anomaly score overlay C Fine-tune DenseNet-121 on the full labelled dataset. Implement Grad-CAM heatmap visualisation on predicted images. Transfer learning, XAI Image upload + heatmap render D Build a radiology report generator: given prediction confidence and detected regions, an LLM produces a 

In [17]:
question = 'What are the four deliverables required for each capstone project?'
answer_question(question)

Answer:
 The four deliverables required for each capstone project are:

1. **Kaggle/Colab Notebooks**: For each sub-problem, there should be reproducible notebooks containing the code used to solve the problem.

2. **Deployed Web or Mobile Application**: Each project must include a running web application or mobile application that can be accessed publicly or downloaded as an APK.

3. **Demo Video**: A 10-minute video demonstrating how the application works and presenting the model results.

4. **Technical Report**: A 4-page IEEE-format technical report detailing the model comparisons and architecture diagrams.

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 1 (similarity 0.455)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 42 (similarity 0.365)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 36 (similarity 0.268)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 11 (similarity 0.262)


('The four deliverables required for each capstone project are:\n\n1. **Kaggle/Colab Notebooks**: For each sub-problem, there should be reproducible notebooks containing the code used to solve the problem.\n\n2. **Deployed Web or Mobile Application**: Each project must include a running web application or mobile application that can be accessed publicly or downloaded as an APK.\n\n3. **Demo Video**: A 10-minute video demonstrating how the application works and presenting the model results.\n\n4. **Technical Report**: A 4-page IEEE-format technical report detailing the model comparisons and architecture diagrams.',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 1,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch AI/ML Capstone Project Problem Statements Parul University in collaboration with TelcoLearn B.Tech 3rd/4th Year — 2027 Graduating Batch — Phase 2 Bootcamp Arpit Tripathi & Sanjay Kumar Instructions for Students

In [18]:
question = 'Which project uses the CICIDS2017 dataset?'
answer_question(question)

Answer:
 The CICIDS2017 dataset is used in the following project:

Project 22: Network Intrusion Detection System  
Domain: Cybersecurity  
Dataset: cikiddataset/cicids2017

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 36 (similarity 0.382)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 42 (similarity 0.372)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 41 (similarity 0.356)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 38 (similarity 0.355)


('The CICIDS2017 dataset is used in the following project:\n\nProject 22: Network Intrusion Detection System  \nDomain: Cybersecurity  \nDataset: cikiddataset/cicids2017',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 36,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch Student Sub-problem T echnique App component A Handle 79 mixed features (36 numeric, 43 categorical). Train Ridge, Lasso, XGBoost. Log-transform price. Target: competitive RMSLE on Kaggle leaderboard. Supervised regression Price prediction API B Cluster properties by location and feature profile using K-Means. Build a neighbourhood value map. Unsupervised segmentation Neighbourhood cluster map C Train a deep tabular model with entity embeddings for categorical features. Implement SHAP for individual price explanation. Deep learning tabular, XAI SHAP feature chart D Given property features and SHAP values, an LLM generates a natural-language valuation 

In [21]:
question = 'Which projects use K-Means clustering for Students sub-problem?'
answer_question(question)

Answer:
 Projects that use K-Means clustering for the Students sub-problem include:

B. Cluster courses by topic and difficulty using TF-IDF + K-Means on course descriptions.
C. Train an LSTM on weekly engagement time series (logins, submissions, forum activity) to predict dropout risk 4 weeks before finals.
D. Given a student’s risk flag and behaviour pattern, an LLM generates a personalised study plan and message from the coordinator.

These projects utilize K-Means clustering within their respective datasets and techniques to analyze and categorize student data effectively.

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 42 (similarity 0.470)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 38 (similarity 0.458)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 21 (similarity 0.418)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 22 (similarity 0.399)


('Projects that use K-Means clustering for the Students sub-problem include:\n\nB. Cluster courses by topic and difficulty using TF-IDF + K-Means on course descriptions.\nC. Train an LSTM on weekly engagement time series (logins, submissions, forum activity) to predict dropout risk 4 weeks before finals.\nD. Given a student’s risk flag and behaviour pattern, an LLM generates a personalised study plan and message from the coordinator.\n\nThese projects utilize K-Means clustering within their respective datasets and techniques to analyze and categorize student data effectively.',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 42,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch LLM API: Google AI Studio (Gemini 1.5 Flash) provides a free tier sufficient for all Student D sub-problems. Deployment: Render.com, Railway.app, and HuggingFace Spaces all offer free hosting tiers for student projects. Page 42 of 42'},
   0.4704

In [22]:
question = 'No'
answer_question(question)

Answer:
 I could not find that in the documents.

Retrieved passages:
[1] AI_ML_Capstone_Projects_2028Batch.pdf, page 41 (similarity 0.150)
[2] AI_ML_Capstone_Projects_2028Batch.pdf, page 24 (similarity 0.140)
[3] AI_ML_Capstone_Projects_2028Batch.pdf, page 26 (similarity 0.129)
[4] AI_ML_Capstone_Projects_2028Batch.pdf, page 35 (similarity 0.124)


('I could not find that in the documents.',
 [({'source': 'AI_ML_Capstone_Projects_2028Batch.pdf',
    'page': 41,
    'text': 'AI/ML Capstone Project Problem Statements Parul University & TelcoLearn — 2027 Batch # Project Title Domain Dataset 20 Student Performance and Early Intervention Education larsen0966/student-performance- data-set 21 Automated Essay Scoring and Feedback Education asap-aes (Kaggle competition) 22 Network Intrusion Detection System Cybersecurity cicdataset/cicids2017 23 Phishing URL Detection Browser Extension Cybersecurity eswarchandt/phishing-website- detector 24 Autonomous Game-Playing Agent RL Gymnasium (CartPole-v1, LunarLander-v2) 25 RL-Based Algorithmic Trading Agent RL + Finance rohanrao/nifty50-stock-market- data 26 Adaptive Traffic Signal Control (RL) RL + Smart City CityFlow open-source simulator 27 AI-Powered Resume Screening and Matching NLP gauravduttakiit/resume-dataset 28 Legal Document Summarisation and Q&A NLP anaderm/indian-court-cases 29 Fake 